# 02b – Reasoning traces with an open model (no API key needed)

Reads `dataset/g50t_dataset.parquet` (from notebook 02), lets an **open vision-language model running on our own GPU** write a natural-language reasoning text for every `play` row, and saves `dataset/g50t_dataset_reasoning.parquet`.

* Model: Qwen2.5-VL-7B-Instruct (open weights, ~16 GB in bf16 → fits the A100).
* The model only **explains** the move; the move itself is always the correct one from the recorded solution.
* Progress is cached in `dataset/reasoning_cache.json` → if it stops, just run the loop cell again.

In [1]:
%pip install -q transformers accelerate datasets pyarrow pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, random, time, torch
from pathlib import Path
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText

DATA = Path("dataset/g50t_dataset.parquet")
OUT = Path("dataset/g50t_dataset_reasoning.parquet")
CACHE = Path("dataset/reasoning_cache.json")
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
NAMES = {"ACTION1": "UP", "ACTION2": "DOWN", "ACTION3": "LEFT", "ACTION4": "RIGHT", "ACTION5": "REWIND"}

d = load_dataset("parquet", data_files=str(DATA), split="train")
play_idx = [i for i, k in enumerate(d["kind"]) if k == "play"]
print(len(d), "rows,", len(play_idx), "play rows to explain")

Generating train split: 0 examples [00:00, ? examples/s]

1716 rows, 286 play rows to explain


## Load the model (first time: downloads ~16 GB, a few minutes)

In [3]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda")
model.eval()
print("GPU memory used:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

GPU memory used: 16.6 GB


In [4]:
def explain(row):
    facts = row["answer"].split("\nAction:")[0]
    a = row["action"]
    prompt = (
        "You are writing a short reasoning trace for a player of the grid puzzle game G50T.\n"
        "Game basics: the player moves one block per action; ACTION5 (REWIND) sends the player back to the start "
        "and the previous path becomes a ghost that repeats it; buttons open gates; the goal is the blue frame.\n"
        f"True facts about this moment: {facts}\n"
        f"The player now chooses {a} ({NAMES[a]}).\n"
        "Write 2-4 sentences in first person: what I see on the board and why this move helps reach the goal. "
        "Only use the facts and the basics above, do not invent new rules. Do not write the action code."
    )
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=[text], images=[row["image"].convert("RGB")], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=180, do_sample=False)
    reply = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    reply = "\n".join(l for l in reply.splitlines() if not l.strip().lower().startswith("action"))
    return reply.strip()

# quick test on one row
r = d[play_idx[5]]
print("FACTS :", r["answer"]); print(); print("MODEL :", explain(r))

FACTS : I am at (2, 1) with 1 ghost(s) replaying. Moves that work here: DOWN, RIGHT, REWIND; blocked: UP, LEFT. Plan for this level: The red/orange button opens its gate only while something stands on it. I walk onto the button, then REWIND so my ghost repeats the walk and stays on the button, while I go through the open gate to the goal. I choose DOWN -> I move from (2, 1) to (2, 2).
Action: ACTION2

MODEL : I see that moving down from (2, 1) to (2, 2) allows me to stand on the red button, which will open its gate. By choosing this move, I can ensure that my ghost will also be able to stand on the button when it repeats the path, allowing both me and the ghost to pass through the gate and reach the goal. This strategy leverages the fact that the button's gate opens only when something is standing on it, ensuring we both can progress towards the blue frame.


## Run for all play rows (≈ 10–30 min). Re-run this cell to resume.

In [5]:
cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}
t0 = time.time()
for n, i in enumerate(play_idx):
    if str(i) in cache: continue
    try:
        cache[str(i)] = explain(d[i])
    except Exception as ex:
        print("error at row", i, repr(ex)); continue
    if n % 10 == 0:
        CACHE.write_text(json.dumps(cache, indent=1))
        print(f"{len(cache)}/{len(play_idx)} done | {time.time()-t0:.0f}s")
CACHE.write_text(json.dumps(cache, indent=1))
print("finished:", len(cache), "/", len(play_idx))

1/286 done | 1s
11/286 done | 18s
21/286 done | 34s
31/286 done | 48s
41/286 done | 62s
51/286 done | 78s
61/286 done | 92s
71/286 done | 107s
81/286 done | 123s
91/286 done | 139s
101/286 done | 155s
111/286 done | 171s
121/286 done | 187s
131/286 done | 202s
141/286 done | 216s
151/286 done | 231s
161/286 done | 246s
171/286 done | 262s
181/286 done | 278s
191/286 done | 293s
201/286 done | 307s
211/286 done | 321s
221/286 done | 335s
231/286 done | 350s
241/286 done | 366s
251/286 done | 380s
261/286 done | 394s
271/286 done | 411s
281/286 done | 427s
finished: 286 / 286


## Save the final dataset

In [6]:
def add_reasoning(row, idx):
    if row["kind"] == "play" and str(idx) in cache:
        row["answer"] = cache[str(idx)] + f"\nAction: {row['action']}"
    return row

d2 = d.map(add_reasoning, with_indices=True)
d2.to_parquet(OUT)
print("saved", OUT, "|", round(OUT.stat().st_size / 1e6, 1), "MB |", len(d2), "rows")
for i in random.sample(play_idx, 3):
    print("\n--- level", d2[i]["level"], "step", d2[i]["step"]); print(d2[i]["answer"])

Map:   0%|          | 0/1716 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

saved dataset/g50t_dataset_reasoning.parquet | 8.2 MB | 1716 rows

--- level 2 step 16
After moving up, I see that the yellow button is now accessible. Triggering this button will open the gate, allowing me to proceed further towards my goal. Since the button only needs to be pressed once, I can safely move forward without leaving any ghosts on it. This move helps me progress by clearing the path and making the next steps possible.
Action: ACTION1

--- level 7 step 29
I see that moving right will take me to a button that opens a gate, allowing me to progress further towards the goal. Since my plan involves using the ghosts to interact with buttons and portals, opening this gate is crucial for advancing. By moving right, I can set up the next steps needed to reach the blue frame efficiently.
Action: ACTION4

--- level 5 step 5
I see that moving right will take me to an area where there's a button and a portal. By going right, I can potentially trigger the button, which might open a gate

## Final check

In [7]:
ok = lambda c, msg: print(("OK   " if c else "FAIL ") + msg)
n_img = sum(1 for x in d2["image"] if x is not None) + sum(1 for x in d2["image_after"] if x is not None)
ok(len(d2) >= 1000, f"rows: {len(d2)}")
ok(n_img >= 1000, f"images: {n_img}")
ok(all(a in NAMES for a in d2["action"]), "all actions valid")
ok(all(x.strip() for x in d2["answer"]), "no empty answers")
ok(all(d2[i]["answer"].endswith(f"Action: {d2[i]['action']}") for i in play_idx), "every play answer ends with its action")
ok(sorted(set(d2["level"])) == list(range(1, 8)), f"levels: {sorted(set(d2['level']))}")
ok(len(cache) == len(play_idx), f"reasoning written for {len(cache)}/{len(play_idx)} play rows")

OK   rows: 1716
OK   images: 3146
OK   all actions valid
OK   no empty answers
OK   every play answer ends with its action
OK   levels: [1, 2, 3, 4, 5, 6, 7]
OK   reasoning written for 286/286 play rows
